# Convolutional Neural Networks

> Based on Stanford CS230 — C4M1, Andrew Ng / DeepLearning.AI

A Convolutional Neural Network (CNN) applies learned filters across an input using a **sliding window**, exploiting spatial locality and parameter sharing to process images far more efficiently than dense layers.

## Learning Objectives

1. Compute output dimensions for convolution, padding, and stride
2. Explain why weight sharing makes CNNs efficient
3. Implement 2D convolution and max-pooling from scratch
4. Trace shapes through a complete conv → pool → flatten → dense pipeline

## Core Formulas

Given input $n \times n$, filter $f \times f$, padding $p$, stride $s$:

$$n_{\text{out}} = \left\lfloor \frac{n + 2p - f}{s} \right\rfloor + 1$$

**Valid conv** ($p=0$): output shrinks.  
**Same conv** ($p = (f-1)/2$): output matches input size.

## Convolution Operation — One Filter

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 680 220" width="680" height="220" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Input 5x5 grid -->
  <text x="60"  y="18" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Input (5×5)</text>
  <g fill="#eef4fb" stroke="#5b9bd5" stroke-width="1">
    <rect x="20"  y="25" width="32" height="32"/><rect x="52"  y="25" width="32" height="32"/>
    <rect x="84"  y="25" width="32" height="32"/><rect x="116" y="25" width="32" height="32"/>
    <rect x="148" y="25" width="32" height="32"/>
    <rect x="20"  y="57" width="32" height="32"/><rect x="52"  y="57" width="32" height="32"/>
    <rect x="84"  y="57" width="32" height="32"/><rect x="116" y="57" width="32" height="32"/>
    <rect x="148" y="57" width="32" height="32"/>
    <rect x="20"  y="89" width="32" height="32"/><rect x="52"  y="89" width="32" height="32"/>
    <rect x="84"  y="89" width="32" height="32"/><rect x="116" y="89" width="32" height="32"/>
    <rect x="148" y="89" width="32" height="32"/>
    <rect x="20"  y="121" width="32" height="32"/><rect x="52"  y="121" width="32" height="32"/>
    <rect x="84"  y="121" width="32" height="32"/><rect x="116" y="121" width="32" height="32"/>
    <rect x="148" y="121" width="32" height="32"/>
    <rect x="20"  y="153" width="32" height="32"/><rect x="52"  y="153" width="32" height="32"/>
    <rect x="84"  y="153" width="32" height="32"/><rect x="116" y="153" width="32" height="32"/>
    <rect x="148" y="153" width="32" height="32"/>
  </g>
  <!-- Highlight 3x3 receptive field -->
  <rect x="20" y="25" width="96" height="96" fill="#5b9bd5" fill-opacity="0.22" stroke="#3a7bbf" stroke-width="2.5" rx="2"/>
  <!-- Numbers in input -->
  <text x="36"  y="46"  text-anchor="middle" fill="#1a1d27" font-size="11">1</text><text x="68"  y="46"  text-anchor="middle" fill="#1a1d27" font-size="11">0</text><text x="100" y="46"  text-anchor="middle" fill="#1a1d27" font-size="11">1</text>
  <text x="36"  y="78"  text-anchor="middle" fill="#1a1d27" font-size="11">0</text><text x="68"  y="78"  text-anchor="middle" fill="#1a1d27" font-size="11">1</text><text x="100" y="78"  text-anchor="middle" fill="#1a1d27" font-size="11">0</text>
  <text x="36"  y="110" text-anchor="middle" fill="#1a1d27" font-size="11">1</text><text x="68"  y="110" text-anchor="middle" fill="#1a1d27" font-size="11">0</text><text x="100" y="110" text-anchor="middle" fill="#1a1d27" font-size="11">1</text>
  <!-- Filter 3x3 -->
  <text x="280" y="18" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Filter (3×3)</text>
  <g fill="#fef6e4" stroke="#f4b942" stroke-width="1.5">
    <rect x="216" y="25" width="44" height="44"/><rect x="260" y="25" width="44" height="44"/><rect x="304" y="25" width="44" height="44"/>
    <rect x="216" y="69" width="44" height="44"/><rect x="260" y="69" width="44" height="44"/><rect x="304" y="69" width="44" height="44"/>
    <rect x="216" y="113" width="44" height="44"/><rect x="260" y="113" width="44" height="44"/><rect x="304" y="113" width="44" height="44"/>
  </g>
  <text x="238" y="52"  text-anchor="middle" fill="#a07010" font-size="13" font-weight="bold">1</text>
  <text x="282" y="52"  text-anchor="middle" fill="#a07010" font-size="13" font-weight="bold">0</text>
  <text x="326" y="52"  text-anchor="middle" fill="#a07010" font-size="13" font-weight="bold">-1</text>
  <text x="238" y="96"  text-anchor="middle" fill="#a07010" font-size="13" font-weight="bold">1</text>
  <text x="282" y="96"  text-anchor="middle" fill="#a07010" font-size="13" font-weight="bold">0</text>
  <text x="326" y="96"  text-anchor="middle" fill="#a07010" font-size="13" font-weight="bold">-1</text>
  <text x="238" y="140" text-anchor="middle" fill="#a07010" font-size="13" font-weight="bold">1</text>
  <text x="282" y="140" text-anchor="middle" fill="#a07010" font-size="13" font-weight="bold">0</text>
  <text x="326" y="140" text-anchor="middle" fill="#a07010" font-size="13" font-weight="bold">-1</text>
  <!-- Arrow -->
  <text x="380" y="95" text-anchor="middle" fill="#555" font-size="22">⊛</text>
  <!-- Output 3x3 grid -->
  <text x="510" y="18" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Output (3×3)</text>
  <g fill="#f0fbf5" stroke="#7ecba1" stroke-width="1.5">
    <rect x="430" y="25" width="54" height="54"/><rect x="484" y="25" width="54" height="54"/><rect x="538" y="25" width="54" height="54"/>
    <rect x="430" y="79" width="54" height="54"/><rect x="484" y="79" width="54" height="54"/><rect x="538" y="79" width="54" height="54"/>
    <rect x="430" y="133" width="54" height="54"/><rect x="484" y="133" width="54" height="54"/><rect x="538" y="133" width="54" height="54"/>
  </g>
  <rect x="430" y="25" width="54" height="54" fill="#7ecba1" fill-opacity="0.3" stroke="#5aab81" stroke-width="2.5" rx="2"/>
  <text x="457" y="58"  text-anchor="middle" fill="#3d7a5e" font-size="13" font-weight="bold">4</text>
  <!-- Labels below -->
  <text x="60"  y="200" text-anchor="middle" fill="#888" font-size="10">n=5, f=3, p=0, s=1</text>
  <text x="510" y="200" text-anchor="middle" fill="#888" font-size="10">n_out = ⌊(5+0−3)/1⌋+1 = 3</text>
</svg>


## Pooling and Volume Notation

For a **volume** of shape $n_H \times n_W \times n_C$ (height × width × channels), each of $n_C'$ filters produces one output channel:

| Layer type | Output shape | Parameters |
|---|---|---|
| Conv ($f$, $p$, $s$, $n_C'$ filters) | $n_{H,\text{out}} \times n_{W,\text{out}} \times n_C'$ | $(f \cdot f \cdot n_C + 1) \cdot n_C'$ |
| Max Pool ($f$, $s$) | $n_{H,\text{out}} \times n_{W,\text{out}} \times n_C$ | **0** (no learnable params) |
| Avg Pool ($f$, $s$) | $n_{H,\text{out}} \times n_{W,\text{out}} \times n_C$ | **0** |
| Flatten | $n_H \cdot n_W \cdot n_C$ | 0 |

**Why CNNs are parameter-efficient:** one $3\!\times\!3$ filter is reused at every spatial position — instead of one weight per pixel per neuron.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- 2D convolution from scratch ----
def conv2d(X, W, b, stride=1, pad=0):
    """Naïve 2D cross-correlation (single filter, single image)."""
    if pad:
        X = np.pad(X, pad, mode='constant')
    H_in, W_in = X.shape
    fh, fw = W.shape
    H_out = (H_in - fh) // stride + 1
    W_out = (W_in - fw) // stride + 1
    Z = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            Z[i, j] = np.sum(X[i*stride:i*stride+fh, j*stride:j*stride+fw] * W) + b
    return Z

def maxpool2d(X, f=2, stride=2):
    H_in, W_in = X.shape
    H_out = (H_in - f) // stride + 1
    W_out = (W_in - f) // stride + 1
    Z = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            Z[i, j] = X[i*stride:i*stride+f, j*stride:j*stride+f].max()
    return Z

# ---- Create a simple test image ----
np.random.seed(0)
img = np.zeros((12, 12))
img[2:5, 2:5] = 1    # bright square top-left
img[7:10, 7:10] = 1  # bright square bottom-right
img += np.random.randn(12, 12) * 0.1

# Edge-detection filters
filters = {
    'Vertical edge':   np.array([[ 1, 0,-1],[ 1, 0,-1],[ 1, 0,-1]], dtype=float),
    'Horizontal edge': np.array([[ 1, 1, 1],[ 0, 0, 0],[-1,-1,-1]], dtype=float),
    'Sobel':           np.array([[-1,-2,-1],[ 0, 0, 0],[ 1, 2, 1]], dtype=float),
    'Sharpening':      np.array([[ 0,-1, 0],[-1, 5,-1],[ 0,-1, 0]], dtype=float),
}

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
fig.suptitle('2D Convolution — Filter Effects', fontsize=13, fontweight='bold')

axes[0][0].imshow(img, cmap='gray', vmin=-0.5, vmax=1.5)
axes[0][0].set_title('Input image (12×12)')
axes[0][0].axis('off')
axes[1][0].axis('off')

for col, (name, filt) in enumerate(filters.items(), start=1):
    out = conv2d(img, filt, b=0, stride=1, pad=0)
    pool_out = maxpool2d(np.maximum(0, out), f=2, stride=2)

    axes[0][col].imshow(out, cmap='RdBu_r')
    axes[0][col].set_title(f'{name}\n→ {out.shape}', fontsize=9)
    axes[0][col].axis('off')

    axes[1][col].imshow(pool_out, cmap='YlGn')
    axes[1][col].set_title(f'After ReLU + MaxPool\n→ {pool_out.shape}', fontsize=9)
    axes[1][col].axis('off')

plt.tight_layout()
plt.show()

# ---- Output size calculator ----
def output_size(n, f, p, s):
    return (n + 2*p - f) // s + 1

print("Output size calculator  n_out = ⌊(n + 2p − f) / s⌋ + 1")
print(f"{'n':>4} {'f':>3} {'p':>3} {'s':>3}  → {'n_out':>6}")
print("-" * 30)
for n, f, p, s in [(28,3,0,1),(28,3,1,1),(28,5,2,1),(28,3,0,2),(224,3,1,1),(224,3,1,2)]:
    print(f"{n:>4} {f:>3} {p:>3} {s:>3}  → {output_size(n,f,p,s):>6}")
